In [1]:
!JAVA -version

openjdk version "21.0.2" 2024-01-16
OpenJDK Runtime Environment (build 21.0.2+13-58)
OpenJDK 64-Bit Server VM (build 21.0.2+13-58, mixed mode, sharing)


In [4]:
!uv pip install pyspark

Resolved 2 packages in 11.48s
Prepared 2 packages in 29.59s
Installed 2 packages in 258ms
 + py4j==0.10.9.9
 + pyspark==4.1.1
Audited 1 package in 10.89s


In [5]:
import pyspark
print(pyspark.__version__)

4.1.1


In [6]:
from pyspark.sql import SparkSession

In [7]:
from pyspark import SparkContext

In [9]:
import os
import sys
from pyspark.sql import SparkSession

# 1. 현재 사용 중인 uv 가상환경의 파이썬 경로를 환경 변수에 등록
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 2. (선택 사항) 만약 JAVA_HOME이 여전히 불안하다면 여기서 한 번 더 고정
os.environ['JAVA_HOME'] = r'C:\openjdk-21.0.2_windows-x64_bin\jdk-21.0.2'

# 3. 세션 생성
try:
    spark = SparkSession.builder \
        .appName("PySpark WordCount Test") \
        .master("local[*]") \
        .getOrCreate()
    print("✨ Spark 세션 생성 성공!")
except Exception as e:
    print(f"❌ 에러 발생: {e}")

❌ 에러 발생: [WinError 2] 지정된 파일을 찾을 수 없습니다


In [10]:
os.environ['PYSPARK_PYTHON'] = 'python'

In [11]:
spark = SparkSession.builder.appName("PySpark WordCount Test").master("local[*]").getOrCreate()

FileNotFoundError: [WinError 2] 지정된 파일을 찾을 수 없습니다

In [12]:
import os
import sys
from pyspark.sql import SparkSession

# 1. 아까 에러에서 확인된 파이썬 실행 파일의 전체 경로를 직접 넣습니다.
# r을 붙여야 역슬래시(\)가 경로로 제대로 인식됩니다.
my_python_path = r"C:\Users\김하영\AppData\Local\uv\cache\builds-v0\.tmpNmfVcn\Scripts\python.exe"

os.environ['PYSPARK_PYTHON'] = my_python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = my_python_path

# 2. Java 경로도 다시 한 번 확실히 고정 (한글 없는 경로!)
os.environ['JAVA_HOME'] = r'C:\openjdk-21.0.2_windows-x64_bin\jdk-21.0.2'

# 3. 세션 생성
try:
    # 기존 세션이 있다면 종료하고 새로 만듭니다.
    if 'spark' in locals():
        spark.stop()
        
    spark = SparkSession.builder \
        .appName("PySpark Final Test") \
        .master("local[1]") \
        .getOrCreate()
    
    print("✨ 드디어 Spark 세션 생성에 성공했습니다!")
    
    # 4. 즉시 테스트 연산
    sc = spark.sparkContext
    test_data = sc.parallelize([1, 2, 3, 4, 5])
    print("✅ 테스트 결과:", test_data.map(lambda x: x * 10).collect())

except Exception as e:
    print(f"❌ 여전히 에러 발생: {e}")

❌ 여전히 에러 발생: [WinError 2] 지정된 파일을 찾을 수 없습니다


In [13]:
import sys
print(sys.executable)

C:\Users\김하영\AppData\Local\uv\cache\builds-v0\.tmpNmfVcn\Scripts\python.exe


In [11]:
data = ["hello spark", "hello docker", "hello worker", "hello world"]

In [12]:
rdd = spark.sparkContext.textFile("/opt/spark/works/sample.text")

In [14]:
sc = spark.sparkContext
rdd = sc.parallelize(["hello world", "pyspark is fun"])

# 1번 테스트 (map 단계)
try:
    words = rdd.flatMap(lambda x: x.split(" "))
    pairs = words.map(lambda x: (x, 1))
    print("성공 데이터:", pairs.take(5))
except Exception as e:
    print("여전히 에러가 발생합니다. 메시지:", e)

여전히 에러가 발생합니다. 메시지: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (121.173.11.198 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRu

In [23]:
for word, count in counts.collect():
    print(f"{word}: {count}")

Apache: 1
Spark: 1
test: 1
: 3
FXking: 1
dififcultasdf: 1
file: 1
example: 1
sample: 1
apache: 1


In [24]:
df = counts.toDF(["word", "count"])

In [25]:
print(df)

DataFrame[word: string, count: bigint]


In [18]:
df.show()

+------+-----+
|  word|count|
+------+-----+
| spark|    1|
|worker|    1|
|docker|    1|
| hello|    3|
+------+-----+



In [30]:
df.coalesce(1).write.mode("overwrite").csv("/opt/spark/works/output/wordcount", header =True)

In [32]:
df.coalesce(1).write.mode("overwrite").json("/opt/spark/works/output/wordcount_json")

In [34]:
spark.stop()